# DUCK+: Temporally-Aware Rumour Detection — PHEME Dataset
**CS 6320 — Kishan Rakesh & Annie Johnson Porattoor**

**Before running:** Runtime → Change runtime type → **A100 GPU** + **High-RAM**

Run cells top-to-bottom. Cells 1–4 are one-time setup. Cell 5 onward are the experiments.

> **PHEME differences vs Twitter15/16:**
> - Cell 4 uploads the PHEME archive and runs `preprocess_pheme.py`
> - Cell 6 uses `pheme` as the dataset with `loeo` and `chrono` splits
> - User features (`userx`) are **real** — embedded in PHEME JSON, no API needed
> - LOEO = Leave-One-Event-Out (9 events → 9 folds, standard PHEME protocol)

## Cell 1 — Anti-idle (run this first, keep this tab open)
Injects a JavaScript keepalive into the browser page so Colab never sees the session as idle. This must be run before anything else.

In [ ]:
# Prevent Colab from disconnecting due to inactivity.
# Clicks the 'Connect' button every 60 seconds via JavaScript.
from IPython.display import Javascript, display

display(Javascript('''
var keepAliveTimer = setInterval(function() {
    console.log('keepAlive ping: ' + new Date().toLocaleTimeString());
    var btn = document.querySelector('colab-connect-button');
    if (btn) {
        var inner = btn.shadowRoot ? btn.shadowRoot.querySelector('button')
                                   : btn.querySelector('button');
        if (inner) inner.click();
    }
}, 60000);
console.log('Anti-idle keepalive started.');
'''))
print('Anti-idle keepalive active. Keep this browser tab open.')


## Cell 2 — Clone repo and set working directory


In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/AnnieJP/duck-rumor-detection'
BRANCH   = 'kishan/duck-plus'
CODE_DIR = '/content/duck-rumor-detection'

if os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, CODE_DIR], check=True)

os.chdir(CODE_DIR)
print('Working dir:', os.getcwd())
print('Branch:', BRANCH)
print('Data files:', sum(len(fs) for _, _, fs in os.walk('data')))


## Cell 3 — Install dependencies
Skip after first run if packages are already cached in the session.

In [ ]:
import subprocess, torch

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

# Colab ships PyTorch — detect version and pick matching PyG wheels.
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.5.1'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')[:3]  # e.g. 'cu121'
print(f'Colab torch={torch_ver}  cuda_tag={cuda_tag}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
print(f'PyG wheel index: {pyg_url}')

print('Installing PyG sparse deps...')
run(f'pip install -q torch-scatter torch-sparse -f {pyg_url}')

print('Installing remaining packages...')
run('pip install -q torch-geometric transformers scikit-learn tqdm numpy pandas networkx scipy')

print('All packages installed.')
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## Cell 4 — Mount Google Drive and preprocess PHEME → .npz files

**One-time setup:**
1. Upload `PHEME_veracity.tar.bz2` to your Google Drive at `My Drive/pheme/PHEME_veracity.tar.bz2`
2. Run this cell — it mounts Drive, extracts, preprocesses, and saves the npz files back to Drive
3. On future sessions it detects the npz files already exist and skips preprocessing instantly

**Share the Drive folder with Annie** so you both use the same preprocessed data.

In [ ]:
import os, subprocess, tarfile, shutil
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_DIR    = '/content/drive/MyDrive/pheme'
ARCHIVE_PATH = f'{DRIVE_DIR}/PHEME_veracity.tar.bz2'
DRIVE_NPZ    = f'{DRIVE_DIR}/pheme_npz'
DRIVE_LOEO   = f'{DRIVE_DIR}/pheme_loeo'
DRIVE_CHRONO = f'{DRIVE_DIR}/pheme_chrono'

LOCAL_NPZ    = 'data/pheme_npz'
LOCAL_LOEO   = 'data/pheme_loeo'
LOCAL_CHRONO = 'data/pheme_chrono'
EXTRACT_DIR  = '/content/pheme_raw'
PHEME_ROOT   = f'{EXTRACT_DIR}/all-rnr-annotated-threads'

os.makedirs('data', exist_ok=True)

already_preprocessed = (
    os.path.isdir(DRIVE_NPZ) and len(os.listdir(DRIVE_NPZ)) > 100
)

if already_preprocessed:
    print(f'Preprocessed data found on Drive ({len(os.listdir(DRIVE_NPZ))} threads) — symlinking.')
else:
    if not os.path.exists(ARCHIVE_PATH):
        raise FileNotFoundError(
            f'Archive not found at {ARCHIVE_PATH}.\n'
            f'Upload PHEME_veracity.tar.bz2 to Google Drive → My Drive/pheme/'
        )

    # Detect actual compression (file is gzip despite .bz2 extension)
    import subprocess as _sp
    magic = _sp.run(['file', ARCHIVE_PATH], capture_output=True, text=True).stdout
    if 'gzip' in magic:
        mode = 'r:gz'
    elif 'bzip2' in magic:
        mode = 'r:bz2'
    else:
        mode = 'r:*'   # let tarfile autodetect
    print(f'Detected archive format: {mode}  ({magic.strip()})')

    print('Extracting archive from Drive...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(ARCHIVE_PATH, mode) as tar:
        tar.extractall(EXTRACT_DIR)
    print('Extraction complete.')

    print('Running preprocess_pheme.py → saving to Drive...')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    result = subprocess.run(
        ['python', 'preprocess_pheme.py',
         '--pheme-root', PHEME_ROOT,
         '--out-root', DRIVE_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('preprocess_pheme.py failed.')

    print(f'Saved {len(os.listdir(DRIVE_NPZ))} threads to Drive.')

# Symlink Drive folders into local data/
for drive_path, local_path in [
    (DRIVE_NPZ,    LOCAL_NPZ),
    (DRIVE_LOEO,   LOCAL_LOEO),
    (DRIVE_CHRONO, LOCAL_CHRONO),
]:
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f'Linked {local_path} → {drive_path}')

print(f'\nReady: {len(os.listdir(LOCAL_NPZ))} threads available at {LOCAL_NPZ}')

## Cell 5 — GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → GPU.')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch version: {torch.__version__}')

## Cell 6 — Configure experiment
Edit the variables in this cell to choose what to run.

PHEME splits:
- `loeo` — Leave-One-Event-Out (9 folds, one per news event). Standard protocol for PHEME comparisons.
- `chrono` — Chronological 60/20/20 split by source tweet timestamp.

In [ ]:
VARIANTS = ['baseline', 'temp', 'gated', 'full']
SPLITS   = ['loeo', 'chrono']

# LOEO: 9 events; chrono: 1 run (fixed split)
LOEO_EVENTS = [
    'charliehebdo', 'ebola-essien', 'ferguson', 'germanwings-crash',
    'gurlitt', 'ottawashooting', 'prince-toronto', 'putinmissing', 'sydneysiege'
]

# ── Hyperparameters ───────────────────────────────────────────────────────────
HID_FEATS    = 64
CT_OUT       = 64
UT_OUT       = 64
LR_BERT      = 2e-5
LR_OTHER     = 1e-3
WEIGHT_DECAY = 5e-5
N_EPOCHS     = 50
BATCH_SIZE   = 8
PATIENCE     = 10
NUM_WORKERS  = 2

RESULT_CSV = 'results/pheme_results.csv'
CKPT_DIR   = 'checkpoints/pheme'

# Build job list: for loeo, fold = event index; for chrono, fold = 0
jobs = []
for variant in VARIANTS:
    for split in SPLITS:
        if split == 'loeo':
            for i, event in enumerate(LOEO_EVENTS):
                jobs.append((variant, 'pheme', split, i, event))
        else:
            jobs.append((variant, 'pheme', split, 0, None))

print(f'Total jobs: {len(jobs)}')
print('First 4:', [(v, d, s, i) for v, d, s, i, _ in jobs[:4]])

## Cell 6b — Stage preset (recommended)
Choose a stage to progressively scale up. `smoke` runs 1 epoch on 16 threads to verify the pipeline end-to-end before committing GPU time.

In [ ]:
STAGE = 'smoke'  # one of: smoke, mini, realistic, full

PRESETS = {
    'smoke': {
        'variants': ['full'],
        'splits': ['loeo'],
        'loeo_events': ['charliehebdo'],   # 1 event only
        'n_epochs': 1,
        'batch_size': 4,
        'smoke_n': 16,
        'result_csv': 'results/pheme_smoke.csv',
        'ckpt_dir': 'checkpoints/pheme_smoke',
    },
    'mini': {
        'variants': ['full'],
        'splits': ['loeo'],
        'loeo_events': ['charliehebdo'],
        'n_epochs': 3,
        'batch_size': 4,
        'smoke_n': 64,
        'result_csv': 'results/pheme_mini.csv',
        'ckpt_dir': 'checkpoints/pheme_mini',
    },
    'realistic': {
        'variants': ['full'],
        'splits': ['loeo', 'chrono'],
        'loeo_events': LOEO_EVENTS,
        'n_epochs': 50,
        'batch_size': 8,
        'smoke_n': None,
        'result_csv': 'results/pheme_realistic.csv',
        'ckpt_dir': 'checkpoints/pheme_realistic',
    },
    'full': {
        'variants': ['baseline', 'temp', 'gated', 'full'],
        'splits': ['loeo', 'chrono'],
        'loeo_events': LOEO_EVENTS,
        'n_epochs': 50,
        'batch_size': 8,
        'smoke_n': None,
        'result_csv': 'results/pheme_results.csv',
        'ckpt_dir': 'checkpoints/pheme',
    },
}

cfg = PRESETS[STAGE]
VARIANTS   = cfg['variants']
SPLITS     = cfg['splits']
N_EPOCHS   = cfg['n_epochs']
BATCH_SIZE = cfg['batch_size']
SMOKE_N    = cfg['smoke_n']
RESULT_CSV = cfg['result_csv']
CKPT_DIR   = cfg['ckpt_dir']

jobs = []
for variant in VARIANTS:
    for split in SPLITS:
        if split == 'loeo':
            for i, event in enumerate(cfg['loeo_events']):
                jobs.append((variant, 'pheme', split, i, event))
        else:
            jobs.append((variant, 'pheme', split, 0, None))

print(f'Stage: {STAGE}  |  Total jobs: {len(jobs)}')
print(f'Variants: {VARIANTS}  |  Splits: {SPLITS}')
print(f'Epochs: {N_EPOCHS}  |  Batch: {BATCH_SIZE}  |  smoke_n: {SMOKE_N}')
print(f'Results -> {RESULT_CSV}')

## Cell 7 — Run all experiments
Runs every job sequentially. Results are appended to `results/results.csv` after each run so progress is saved even if the session disconnects.

In [ ]:
import os, sys, time, csv, pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

sys.path.insert(0, os.path.join(os.getcwd(), 'model'))
from dataset import DuckPlusDataset
from duck_plus import DuckPlus
from train_duck_plus import set_seed, EarlyStopping, evaluate

device = torch.device('cuda:0')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)

NPZ_DIR = 'data/pheme_npz'

# PHEME-specific result columns include 'event'
PHEME_RESULT_COLS = [
    'variant', 'dataset', 'split', 'run', 'event',
    'test_acc', 'test_macro_f1',
    'f1_NR', 'f1_FR', 'f1_TR', 'f1_UR',
    'val_f1', 'val_acc',
]

def append_pheme_result(result, csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=PHEME_RESULT_COLS)
        if write_header:
            writer.writeheader()
        writer.writerow({k: result.get(k, '') for k in PHEME_RESULT_COLS})

# Resume support — key includes event to avoid collision between LOEO folds
completed = set()
if os.path.exists(RESULT_CSV):
    with open(RESULT_CSV) as f:
        for row in csv.DictReader(f):
            completed.add((row['variant'], row['split'], row['run'], row.get('event', '')))
    print(f'Found {len(completed)} already-completed runs — skipping them.')

total   = len(jobs)
done    = 0
t_start = time.time()

for variant, dataset, split, fold, event in jobs:
    key = (variant, split, str(fold), event or '')
    if key in completed:
        print(f'  SKIP (already done): variant={variant} split={split} event={event}')
        done += 1
        continue

    label = event if event else 'chrono'
    print(f'\n[{done+1}/{total}] variant={variant} split={split} fold={label}')

    set_seed(42 + fold)

    # Load split IDs
    if split == 'loeo':
        fold_dir = f'data/pheme_loeo/event_{event}'
        with open(f'{fold_dir}/train.pkl', 'rb') as f_: train_ids = pickle.load(f_)
        with open(f'{fold_dir}/test.pkl',  'rb') as f_: test_ids  = pickle.load(f_)
        val_ids = test_ids
    else:
        with open('data/pheme_chrono/train.pkl', 'rb') as f_: train_ids = pickle.load(f_)
        with open('data/pheme_chrono/val.pkl',   'rb') as f_: val_ids   = pickle.load(f_)
        with open('data/pheme_chrono/test.pkl',  'rb') as f_: test_ids  = pickle.load(f_)

    if SMOKE_N is not None:
        train_ids = train_ids[:SMOKE_N]
        val_ids   = val_ids[:SMOKE_N]
        test_ids  = test_ids[:SMOKE_N]

    train_loader = DataLoader(DuckPlusDataset(train_ids, NPZ_DIR),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS)
    val_loader   = DataLoader(DuckPlusDataset(val_ids,   NPZ_DIR),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS)
    test_loader  = DataLoader(DuckPlusDataset(test_ids,  NPZ_DIR),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS)

    model = DuckPlus(
        variant=variant, hid_feats=HID_FEATS,
        ct_out=CT_OUT, ut_out=UT_OUT, num_classes=4,
    ).to(device)

    bert_ids = set()
    for branch in ['comment_tree', 'comment_chain', 'user_tree']:
        m = getattr(model, branch, None)
        if m and hasattr(m, 'bert'):
            bert_ids.update(id(p) for p in m.bert.parameters())
    optimizer = torch.optim.Adam([
        {'params': [p for p in model.parameters() if id(p) in bert_ids],     'lr': LR_BERT},
        {'params': [p for p in model.parameters() if id(p) not in bert_ids], 'lr': LR_OTHER},
    ], weight_decay=WEIGHT_DECAY)

    ckpt_path = f'{CKPT_DIR}/{variant}_pheme_{split}_r{fold}.pt'
    stopper   = EarlyStopping(patience=PATIENCE, ckpt_path=ckpt_path)

    for epoch in range(N_EPOCHS):
        model.train()
        tr_loss, tr_correct, tr_total = [], 0, 0
        for batch in train_loader:
            batch = batch.to(device)
            logits, _ = model(batch)
            labels = batch.y.view(-1)
            loss = F.nll_loss(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss.append(loss.item())
            tr_correct += logits.argmax(-1).eq(labels).sum().item()
            tr_total   += labels.size(0)

        val_loss, val_acc, val_f1, val_f1_cls = evaluate(model, val_loader, device)
        tr_acc = tr_correct / max(tr_total, 1)
        print(f'  Ep {epoch:03d} tr_loss={np.mean(tr_loss):.3f} tr_acc={tr_acc:.3f} '
              f'val_loss={val_loss:.3f} val_f1={val_f1:.3f}')

        stopper(val_f1, val_acc, val_f1_cls, model)
        if stopper.early_stop:
            print(f'  Early stop at epoch {epoch}.')
            break
        torch.cuda.empty_cache()

    model.load_state_dict(torch.load(ckpt_path, map_location=device,
                                     weights_only=True))
    _, test_acc, test_f1, test_f1_cls = evaluate(model, test_loader, device)

    result = {
        'variant': variant, 'dataset': 'pheme', 'split': split,
        'run': fold, 'event': event or 'chrono',
        'test_acc': round(test_acc, 4), 'test_macro_f1': round(test_f1, 4),
        'f1_NR': round(test_f1_cls[0], 4), 'f1_FR': round(test_f1_cls[1], 4),
        'f1_TR': round(test_f1_cls[2], 4), 'f1_UR': round(test_f1_cls[3], 4),
        'val_f1': round(stopper.best_f1, 4), 'val_acc': round(stopper.best_acc, 4),
    }
    append_pheme_result(result, RESULT_CSV)
    completed.add(key)
    done += 1

    elapsed     = (time.time() - t_start) / 60
    avg_per_job = elapsed / done if done else 0
    print(f'  TEST acc={test_acc:.4f} macro-F1={test_f1:.4f} '
          f'NR={test_f1_cls[0]:.4f} FR={test_f1_cls[1]:.4f} '
          f'TR={test_f1_cls[2]:.4f} UR={test_f1_cls[3]:.4f}')
    print(f'  Progress: {done}/{total} | ~{avg_per_job*(total-done):.0f} min remaining')

    del model, optimizer
    torch.cuda.empty_cache()

print(f'\nAll done! Results saved to {RESULT_CSV}')

## Cell 8 — Aggregate and display results table

In [ ]:
import pandas as pd

df = pd.read_csv(RESULT_CSV)
print(f'Total rows: {len(df)}\n')

# LOEO summary: average across all 9 events per variant
print('=== LOEO Results (mean ± std across events) ===')
loeo = df[df['split'] == 'loeo']
if not loeo.empty:
    summary_loeo = (
        loeo.groupby('variant')
            [['test_macro_f1', 'test_acc', 'f1_NR', 'f1_FR', 'f1_TR', 'f1_UR']]
            .agg(['mean', 'std'])
            .round(4)
    )
    print(summary_loeo.to_string())

print('\n=== Chronological Results ===')
chrono = df[df['split'] == 'chrono']
if not chrono.empty:
    summary_chrono = (
        chrono.groupby('variant')
              [['test_macro_f1', 'test_acc', 'f1_NR', 'f1_FR', 'f1_TR', 'f1_UR']]
              .round(4)
    )
    print(summary_chrono.to_string())

print('\n=== Per-event LOEO breakdown ===')
if not loeo.empty:
    per_event = (
        loeo.groupby(['variant', 'event'])['test_macro_f1']
            .mean()
            .unstack('event')
            .round(4)
    )
    print(per_event.to_string())

## Cell 9 — Plot: macro-F1 by variant and split type

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(RESULT_CSV)
variant_order = ['baseline', 'temp', 'gated', 'full']
colors = {'loeo': '#4C72B0', 'chrono': '#DD8452'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: LOEO macro-F1 per variant (mean ± std across 9 events)
ax = axes[0]
loeo = df[df['split'] == 'loeo']
if not loeo.empty:
    means = loeo.groupby('variant')['test_macro_f1'].mean().reindex(variant_order)
    stds  = loeo.groupby('variant')['test_macro_f1'].std().reindex(variant_order)
    x = np.arange(len(variant_order))
    ax.bar(x, means, 0.5, yerr=stds, capsize=5,
           color=colors['loeo'], alpha=0.85, label='LOEO')
ax.set_title('PHEME — LOEO (avg over 9 events)', fontsize=13)
ax.set_xticks(np.arange(len(variant_order)))
ax.set_xticklabels(variant_order)
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Right: Chronological macro-F1 per variant
ax = axes[1]
chrono = df[df['split'] == 'chrono']
if not chrono.empty:
    means_c = chrono.groupby('variant')['test_macro_f1'].mean().reindex(variant_order)
    ax.bar(np.arange(len(variant_order)), means_c, 0.5,
           color=colors['chrono'], alpha=0.85, label='Chrono')
ax.set_title('PHEME — Chronological (60/20/20)', fontsize=13)
ax.set_xticks(np.arange(len(variant_order)))
ax.set_xticklabels(variant_order)
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('DUCK+ Ablation on PHEME — Macro-F1 by Variant', fontsize=14)
plt.tight_layout()
plt.savefig('results/pheme_macro_f1_plot.png', dpi=150)
plt.show()
print('Saved to results/pheme_macro_f1_plot.png')

## Cell 10 — Inspect learned gate values (full variant only)

In [ ]:
import sys, os, pickle, torch
import numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), 'model'))
from dataset import DuckPlusDataset
from duck_plus import DuckPlus
from torch_geometric.loader import DataLoader

# Uses the first LOEO fold (charliehebdo held out) by default
GATE_EVENT = 'charliehebdo'
GATE_FOLD  = 0
ckpt_path  = f'{CKPT_DIR}/full_pheme_loeo_r{GATE_FOLD}.pt'

if not os.path.exists(ckpt_path):
    print(f'Checkpoint not found: {ckpt_path}. Run Cell 7 first.')
else:
    with open(f'data/pheme_loeo/event_{GATE_EVENT}/test.pkl', 'rb') as f:
        test_ids = pickle.load(f)

    loader = DataLoader(
        DuckPlusDataset(test_ids, 'data/pheme_npz'),
        batch_size=8, shuffle=False, num_workers=2
    )
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    model = DuckPlus(
        variant='full', hid_feats=HID_FEATS, ct_out=CT_OUT, ut_out=UT_OUT,
    ).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    all_gates = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, gates = model(batch)
            if gates is not None:
                all_gates.append(gates.cpu().numpy())

    all_gates = np.concatenate(all_gates, axis=0)
    branch_names = ['comment_tree (g1)', 'comment_chain (g2)', 'user_tree (g3)']
    print(f'Gate statistics — {GATE_EVENT} test set ({len(all_gates)} threads):')
    print(f'  {"Branch":<25} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
    print('  ' + '-'*57)
    for i, name in enumerate(branch_names):
        g = all_gates[:, i]
        print(f'  {name:<25} {g.mean():8.4f} {g.std():8.4f} '
              f'{g.min():8.4f} {g.max():8.4f}')
    print()
    print('Note: on PHEME, user_tree (g3) should be non-trivial since userx is real.')